# DoorDash Delivery ETA — Etapa 05: Auditoria de Features e Validação Temporal

Este notebook realiza:
1. Auditoria completa das colunas de `df_feat` (originais vs derivadas, listas de disponibilidade e verificação anti-*leakage*).
2. Auditoria e correção de empates em `historical_store_order_count` (mesmo `store_id` e `created_at`).
3. Auditoria de `item_price_range < 0` e criação da flag `invalid_item_price_range`.
4. Auditoria das razões operacionais (`outstanding_orders_per_onshift_dasher` e `busy_dashers_ratio`).
5. Construção e comparação de estratégias de divisão temporal (`src/temporal_validation.py`): simples vs embargo baseado em `actual_delivery_time`.

> **Regras desta etapa:** Nenhum modelo é treinado, o CSV original permanece intacto e não há clipping nem substituição de valores negativos por zero.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets
from src.feature_engineering import (
    build_features,
    CONFIRMED_FEATURES,
    UNCERTAIN_EXPERIMENTAL_FEATURES,
    FORBIDDEN_LEAKAGE_COLS,
)
from src.temporal_validation import (
    temporal_split_simple,
    temporal_split_with_embargo,
    summarize_split,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
df_raw, _, df_model, _ = prepare_datasets(RAW_PATH)
df_feat = build_features(df_model)

print(f"df_raw: {df_raw.shape} | df_model: {df_model.shape} | df_feat: {df_feat.shape}")

## 1. Auditoria das Features e Controle Anti-Leakage

In [ ]:
all_cols = set(df_feat.columns)
accounted = set(CONFIRMED_FEATURES) | set(UNCERTAIN_EXPERIMENTAL_FEATURES) | set(FORBIDDEN_LEAKAGE_COLS) | {"created_at"}
assert all_cols == accounted, f"Diferença encontrada: {all_cols ^ accounted}"
assert len(set(CONFIRMED_FEATURES) & set(FORBIDDEN_LEAKAGE_COLS)) == 0
assert len(set(UNCERTAIN_EXPERIMENTAL_FEATURES) & set(FORBIDDEN_LEAKAGE_COLS)) == 0
print(f"Auditoria aprovada: todas as {len(all_cols)} colunas de df_feat estão classificadas e livres de vazamento.")

## 2. Auditoria de `historical_store_order_count` (Empates de Timestamp)

In [ ]:
dup_mask = df_feat.duplicated(subset=["store_id", "created_at"], keep=False)
print(f"Registros com mesmo (store_id, created_at): {dup_mask.sum()}")
max_unique_tie = df_feat[dup_mask].groupby(["store_id", "created_at"])["historical_store_order_count"].nunique().max()
print(f"Máximo de contagens distintas por empate (deve ser 1): {max_unique_tie}")

## 3. Auditoria de `item_price_range` e `invalid_item_price_range`

In [ ]:
neg_ipr = df_feat[df_feat["item_price_range"] < 0]
print(f"Registros com item_price_range < 0 (max_item_price < min_item_price): {len(neg_ipr)} ({len(neg_ipr)/len(df_feat)*100:.2f}%)")
print(f"Flag invalid_item_price_range == 1: {int(df_feat['invalid_item_price_range'].sum())}")
display(neg_ipr[["created_at", "market_id", "store_id", "total_items", "subtotal", "min_item_price", "max_item_price", "item_price_range"]].head(10))

## 4. Auditoria das Razões Operacionais

In [ ]:
for col in ["outstanding_orders_per_onshift_dasher", "busy_dashers_ratio"]:
    s = df_feat[col].dropna()
    print(f"=== {col} (n={len(s)}) ===")
    print(f"  > 5: {(s > 5).sum()} | > 10: {(s > 10).sum()} | P99: {s.quantile(0.99):.2f} | Max: {s.max():.2f}")
    display(df_feat.nlargest(5, col)[["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders", col]])

## 5. Comparação de Estratégias de Divisão Temporal

In [ ]:
for label, v_cut, t_cut in [
    ("Corte 00:00 UTC (Pico Noturno EUA)", "2015-02-10 00:00:00", "2015-02-14 00:00:00"),
    ("Corte 10:00 UTC (Janela Sem Pedidos 09-13h UTC)", "2015-02-10 10:00:00", "2015-02-14 10:00:00"),
]:
    print(f"\n================ {label} ================")
    s = temporal_split_simple(df_feat, v_cut, t_cut)
    e = temporal_split_with_embargo(df_feat, v_cut, t_cut)
    print(f"Cruzamentos Train->Val: {len(s.dropped_train_to_val)} | Val->Test: {len(s.dropped_val_to_test)}")
    display(summarize_split(e))